In [1]:
!pip install lightgbm -q

In [2]:
import pandas as pd
import numpy as np

import lightgbm as lgb

from sklearn.metrics import mean_absolute_error, mean_squared_error

print("LightGBM imported successfully")

LightGBM imported successfully


In [3]:
query = """
SELECT
    item_id,
    store_id,
    ds,
    y,
    lag_1,
    lag_7,
    lag_14,
    rolling_7,
    rolling_28,
    day_of_week,
    day_of_month,
    week_of_year,
    month,
    quarter,
    year
FROM
    `retail-forecasting-506113.retail_forecasting.lightgbm_training_data`
ORDER BY
    item_id,
    store_id,
    ds
"""

lgb_df = client.query(query).to_dataframe()

print("Rows:", len(lgb_df))
print("Columns:", lgb_df.columns.tolist())

NameError: name 'client' is not defined

In [4]:
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

client = bigquery.Client(
    project="retail-forecasting-506113"
)

print("BigQuery connected successfully")

BigQuery connected successfully


In [5]:
query = """
SELECT
    item_id,
    store_id,
    ds,
    y,
    lag_1,
    lag_7,
    lag_14,
    rolling_7,
    rolling_28,
    day_of_week,
    day_of_month,
    week_of_year,
    month,
    quarter,
    year
FROM
    `retail-forecasting-506113.retail_forecasting.lightgbm_training_data`
ORDER BY
    item_id,
    store_id,
    ds
"""

lgb_df = client.query(query).to_dataframe()

print("Rows:", len(lgb_df))
print("Columns:", lgb_df.columns.tolist())

Rows: 9705
Columns: ['item_id', 'store_id', 'ds', 'y', 'lag_1', 'lag_7', 'lag_14', 'rolling_7', 'rolling_28', 'day_of_week', 'day_of_month', 'week_of_year', 'month', 'quarter', 'year']


In [6]:
lgb_df = lgb_df.dropna(
    subset=["lag_1", "lag_7", "lag_14"]
).copy()

print("Rows after removing lag nulls:", len(lgb_df))

Rows after removing lag nulls: 9635


In [7]:
lgb_df = lgb_df.sort_values(
    ["item_id", "store_id", "ds"]
).reset_index(drop=True)

lgb_df["series"] = (
    lgb_df["item_id"] + "_" + lgb_df["store_id"]
)

test = (
    lgb_df
    .groupby("series")
    .tail(28)
)

train = lgb_df.drop(test.index)

print("Training rows:", len(train))
print("Test rows:", len(test))
print("Number of series:", lgb_df["series"].nunique())

Training rows: 9495
Test rows: 140
Number of series: 5


In [8]:
from sklearn.preprocessing import LabelEncoder

item_encoder = LabelEncoder()
store_encoder = LabelEncoder()

train["item_encoded"] = item_encoder.fit_transform(train["item_id"])
test["item_encoded"] = item_encoder.transform(test["item_id"])

train["store_encoded"] = store_encoder.fit_transform(train["store_id"])
test["store_encoded"] = store_encoder.transform(test["store_id"])

print("Categorical encoding completed")

Categorical encoding completed


/tmp/ipykernel_982/3523524498.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test["item_encoded"] = item_encoder.transform(test["item_id"])
/tmp/ipykernel_982/3523524498.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test["store_encoded"] = store_encoder.transform(test["store_id"])


In [9]:
from sklearn.preprocessing import LabelEncoder

# Make independent copies
train = train.copy()
test = test.copy()

item_encoder = LabelEncoder()
store_encoder = LabelEncoder()

train["item_encoded"] = item_encoder.fit_transform(train["item_id"])
test["item_encoded"] = item_encoder.transform(test["item_id"])

train["store_encoded"] = store_encoder.fit_transform(train["store_id"])
test["store_encoded"] = store_encoder.transform(test["store_id"])

print("Categorical encoding completed")

Categorical encoding completed


In [10]:
features = [
    "item_encoded",
    "store_encoded",
    "lag_1",
    "lag_7",
    "lag_14",
    "rolling_7",
    "rolling_28",
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "month",
    "quarter",
    "year"
]

X_train = train[features]
y_train = train["y"]

X_test = test[features]
y_test = test["y"]

print("Number of features:", len(features))
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Number of features: 13
X_train shape: (9495, 13)
X_test shape: (140, 13)


In [11]:
import lightgbm as lgb

model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

print("LightGBM training completed.")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000910 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1400
[LightGBM] [Info] Number of data points in the train set: 9495, number of used features: 13
[LightGBM] [Info] Start training from score 89.473723
LightGBM training completed.


In [12]:
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)

rmse = np.sqrt(
    mean_squared_error(y_test, predictions)
)

print("LightGBM MAE:", round(mae, 2))
print("LightGBM RMSE:", round(rmse, 2))

LightGBM MAE: 12.16
LightGBM RMSE: 15.37


In [13]:
lightgbm_forecasts = test[
    ["item_id", "store_id", "ds", "y"]
].copy()

lightgbm_forecasts["predicted_demand"] = predictions

lightgbm_forecasts = lightgbm_forecasts.rename(
    columns={"y": "actual_demand"}
)

lightgbm_forecasts["model"] = "LightGBM"

lightgbm_forecasts = lightgbm_forecasts[
    [
        "item_id",
        "store_id",
        "ds",
        "actual_demand",
        "predicted_demand",
        "model"
    ]
]

print("Forecast rows:", len(lightgbm_forecasts))
print(lightgbm_forecasts.head())

Forecast rows: 140
          item_id store_id          ds  actual_demand  predicted_demand  \
1899  FOODS_3_090     CA_1  2016-04-25           48.0         51.734801   
1900  FOODS_3_090     CA_1  2016-04-26           35.0         45.890840   
1901  FOODS_3_090     CA_1  2016-04-27           34.0         48.650322   
1902  FOODS_3_090     CA_1  2016-04-28           67.0         47.935335   
1903  FOODS_3_090     CA_1  2016-04-29           63.0         64.243745   

         model  
1899  LightGBM  
1900  LightGBM  
1901  LightGBM  
1902  LightGBM  
1903  LightGBM  


In [1]:
from google.cloud import bigquery

table_id = (
    "retail-forecasting-506113."
    "retail_forecasting.lightgbm_forecasts"
)

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(
    lightgbm_forecasts,
    table_id,
    job_config=job_config
)

job.result()

print("LightGBM forecasts uploaded successfully.")

NameError: name 'client' is not defined

In [2]:
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

client = bigquery.Client(
    project="retail-forecasting-506113"
)

print("BigQuery connected successfully")

BigQuery connected successfully


In [3]:
table_id = (
    "retail-forecasting-506113."
    "retail_forecasting.lightgbm_forecasts"
)

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(
    lightgbm_forecasts,
    table_id,
    job_config=job_config
)

job.result()

print("LightGBM forecasts uploaded successfully.")

NameError: name 'lightgbm_forecasts' is not defined

In [4]:
print("lightgbm_forecasts" in globals())

False


In [5]:
lightgbm_forecasts = test[
    ["item_id", "store_id", "ds", "y"]
].copy()

lightgbm_forecasts["predicted_demand"] = predictions

lightgbm_forecasts = lightgbm_forecasts.rename(
    columns={"y": "actual_demand"}
)

lightgbm_forecasts["model"] = "LightGBM"

lightgbm_forecasts = lightgbm_forecasts[
    [
        "item_id",
        "store_id",
        "ds",
        "actual_demand",
        "predicted_demand",
        "model"
    ]
]

print("Forecast rows:", len(lightgbm_forecasts))
print(lightgbm_forecasts.head())

NameError: name 'test' is not defined

In [6]:
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

client = bigquery.Client(
    project="retail-forecasting-506113"
)

print("BigQuery connected successfully")

BigQuery connected successfully


In [7]:
query = """
SELECT
    item_id,
    store_id,
    ds,
    y,
    lag_1,
    lag_7,
    lag_14,
    rolling_7,
    rolling_28,
    day_of_week,
    day_of_month,
    week_of_year,
    month,
    quarter,
    year
FROM
    `retail-forecasting-506113.retail_forecasting.lightgbm_training_data`
ORDER BY
    item_id,
    store_id,
    ds
"""

lgb_df = client.query(query).to_dataframe()

print("Rows:", len(lgb_df))

Rows: 9705


In [8]:
from sklearn.preprocessing import LabelEncoder

lgb_df = lgb_df.dropna(
    subset=["lag_1", "lag_7", "lag_14"]
).copy()

lgb_df = lgb_df.sort_values(
    ["item_id", "store_id", "ds"]
).reset_index(drop=True)

lgb_df["series"] = (
    lgb_df["item_id"] + "_" + lgb_df["store_id"]
)

test = (
    lgb_df
    .groupby("series")
    .tail(28)
)

train = lgb_df.drop(test.index).copy()
test = test.copy()

item_encoder = LabelEncoder()
store_encoder = LabelEncoder()

train["item_encoded"] = item_encoder.fit_transform(train["item_id"])
test["item_encoded"] = item_encoder.transform(test["item_id"])

train["store_encoded"] = store_encoder.fit_transform(train["store_id"])
test["store_encoded"] = store_encoder.transform(test["store_id"])

print("Training rows:", len(train))
print("Test rows:", len(test))

Training rows: 9495
Test rows: 140


In [9]:
features = [
    "item_encoded",
    "store_encoded",
    "lag_1",
    "lag_7",
    "lag_14",
    "rolling_7",
    "rolling_28",
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "month",
    "quarter",
    "year"
]

X_train = train[features]
y_train = train["y"]

X_test = test[features]
y_test = test["y"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (9495, 13)
X_test: (140, 13)


In [10]:
import lightgbm as lgb

model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

predictions = model.predict(X_test)

print("LightGBM training and prediction completed.")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000893 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1400
[LightGBM] [Info] Number of data points in the train set: 9495, number of used features: 13
[LightGBM] [Info] Start training from score 89.473723
LightGBM training and prediction completed.


In [11]:
lightgbm_forecasts = test[
    ["item_id", "store_id", "ds", "y"]
].copy()

lightgbm_forecasts["predicted_demand"] = predictions

lightgbm_forecasts = lightgbm_forecasts.rename(
    columns={"y": "actual_demand"}
)

lightgbm_forecasts["model"] = "LightGBM"

lightgbm_forecasts = lightgbm_forecasts[
    [
        "item_id",
        "store_id",
        "ds",
        "actual_demand",
        "predicted_demand",
        "model"
    ]
]

print("Forecast rows:", len(lightgbm_forecasts))

Forecast rows: 140


In [12]:
table_id = (
    "retail-forecasting-506113."
    "retail_forecasting.lightgbm_forecasts"
)

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(
    lightgbm_forecasts,
    table_id,
    job_config=job_config
)

job.result()

print("LightGBM forecasts uploaded successfully.")

LightGBM forecasts uploaded successfully.
